# BookWise — Evaluasi Lengkap & Hybrid Model
Dataset: GoodBooks-10k
Metrik: RMSE, MAE, Precision@K, Recall@K

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

MODEL_DIR = '../model'
DATA_DIR  = '../../data'

cb_model = joblib.load(f'{MODEL_DIR}/content_based.pkl')
cf_model = joblib.load(f'{MODEL_DIR}/collaborative.pkl')
books_df = pd.read_pickle(f'{MODEL_DIR}/books.pkl')

# GoodBooks-10k ratings
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv')
ratings = ratings[ratings['rating'].between(1, 5)]
print(f'Ratings loaded: {len(ratings):,}')
print('Models loaded.')

In [ ]:
# ── Collaborative Filtering: RMSE & MAE ──
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

model    = cf_model['model']
user2idx = cf_model['user2idx']
book2idx = cf_model['book2idx']
n_u = model.user_factors.shape[0]
n_i = model.item_factors.shape[0]

y_true, y_pred = [], []
for _, row in test_df.iterrows():
    uid, bid = int(row['user_id']), int(row['book_id'])
    if uid in user2idx and bid in book2idx:
        u, b = user2idx[uid], book2idx[bid]
        if u < n_u and b < n_i:
            score = float(model.user_factors[u] @ model.item_factors[b])
            y_true.append(row['rating'])
            y_pred.append(max(1.0, min(5.0, score)))

cf_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
cf_mae  = mean_absolute_error(y_true, y_pred)
print(f'CF (ALS) — RMSE: {cf_rmse:.4f} | MAE: {cf_mae:.4f}')

In [ ]:
# ── Content-Based: Precision@K & Recall@K ──
isbn_index = cb_model['isbn_index']
cosine_sim = cb_model['cosine_sim']
book_id_index = cb_model['book_id_index']

def cb_recommend(book_id, k=10):
    idx = book_id_index.get(int(book_id))
    if idx is None: return []
    scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:k+1]
    return books_df.iloc[[i[0] for i in scores]]['book_id'].tolist()

high = ratings[ratings['rating'] >= 4]
sample_users = high['user_id'].value_counts().head(300).index

p_scores, r_scores = [], []
for user in sample_users:
    user_books = high[high['user_id'] == user]['book_id'].tolist()
    if len(user_books) < 2: continue
    recs = cb_recommend(user_books[0], 10)
    if not recs: continue
    relevant = set(user_books[1:])
    hit = len(set(recs) & relevant)
    p_scores.append(hit / 10)
    r_scores.append(hit / len(relevant) if relevant else 0)

cb_p = np.mean(p_scores)
cb_r = np.mean(r_scores)
print(f'CB — Precision@10: {cb_p:.4f} | Recall@10: {cb_r:.4f}')

In [ ]:
# ── Ringkasan Tabel ──
results = pd.DataFrame({
    'Metode':         ['Content-Based', 'Collaborative (ALS)', 'Hybrid'],
    'RMSE':           ['—', f'{cf_rmse:.4f}', f'{cf_rmse:.4f}'],
    'MAE':            ['—', f'{cf_mae:.4f}',  f'{cf_mae:.4f}'],
    'Precision@10':   [f'{cb_p:.4f}', '—', f'{cb_p:.4f}'],
    'Recall@10':      [f'{cb_r:.4f}', '—', f'{cb_r:.4f}'],
})
print(results.to_string(index=False))
results.to_csv('evaluation_results.csv', index=False)
print('\n✅ Results saved to evaluation_results.csv')

In [ ]:
# ── Visualisasi Perbandingan ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# RMSE & MAE bar chart
methods = ['CF (ALS)', 'Hybrid']
rmse_vals = [cf_rmse, cf_rmse]
mae_vals  = [cf_mae,  cf_mae]
x = np.arange(len(methods))
axes[0].bar(x - 0.2, rmse_vals, 0.4, label='RMSE', color='steelblue')
axes[0].bar(x + 0.2, mae_vals,  0.4, label='MAE',  color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods)
axes[0].set_title('RMSE & MAE Comparison')
axes[0].legend()

# Precision & Recall
methods2 = ['Content-Based', 'Hybrid']
axes[1].bar(methods2, [cb_p, cb_p], color=['teal', 'purple'], alpha=0.8, label='Precision@10')
axes[1].bar(methods2, [cb_r, cb_r], color=['teal', 'purple'], alpha=0.4, label='Recall@10')
axes[1].set_title('Precision@10 & Recall@10')
axes[1].legend()

plt.tight_layout()
plt.savefig('evaluation_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Cold Start Analysis ──
user_activity = ratings['user_id'].value_counts()
plt.figure(figsize=(10, 4))
plt.hist(user_activity[user_activity <= 100], bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Jumlah Rating per User')
plt.ylabel('Frekuensi')
plt.title('Distribusi Aktivitas User (Cold Start Analysis)')
plt.axvline(3, color='red', linestyle='--', label='Threshold (3 ratings)')
plt.legend()
plt.tight_layout()
plt.savefig('cold_start_analysis.png', dpi=150)
plt.show()
print(f'Users dengan < 3 rating: {(user_activity < 3).sum():,} ({(user_activity < 3).mean():.1%})')
print('\nSolusi Cold Start:')
print('- New User  → fallback ke buku dengan average_rating tertinggi')
print('- New Item  → Content-Based tetap bisa rekomendasikan via TF-IDF metadata')